# Sparkle V2 — Fase 3.5a: re-extracción de apariencia a 1.280-d

**Protocolo pre-registrado:** `Protocolo_Exp2_Proyeccion_Apariencia.md` (2026-07-29), §2.

**Qué cambia respecto de 3.2a y qué NO.**

| | 3.2a (actual) | 3.5a (este) |
| :--- | :--- | :--- |
| Backbone | MobileNetV2 α=0,35 fine-tuned en FER2013 | **el mismo, cargado del checkpoint** |
| Recorte facial, muestreo, `preprocess_input` | idénticos | **idénticos** |
| Salida guardada | 128-d (tras proyección **aleatoria congelada**) | **1.280-d crudo**, antes de proyectar |
| dtype | float32 | **float16** (~351 MB vs ~702 MB en Drive) |

El backbone **no se reentrena ni se reconstruye**: se carga de
`checkpoints_fase3_2/cnn_extractor.keras`, que es exactamente el que produjo los embeddings
actuales. Eso mantiene la comparación limpia (regla 8) y habilita el assert de fidelidad de
la sección 5: reproyectar el 1.280-d con los pesos aleatorios guardados tiene que devolver
los 128-d existentes. Si ese assert pasa, el brazo R1 del experimento 2 es **exactamente**
el estado actual, y cualquier diferencia contra R2/R3 es atribuible solo a la proyección.

**Entorno:** Colab con **T4 GPU** + credenciales de Kaggle. ~1 h 45. Es reanudable.

In [ ]:
# mediapipe es lo unico que Colab no trae. TensorFlow/Keras/OpenCV ya estan: NO reinstalar.
%pip install -q mediapipe kagglehub

In [ ]:
# =====================================================================
# SETUP — idéntico a 3.2a salvo los directorios de salida y el dtype
# =====================================================================
import sys, os, glob, json, time, shutil

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR, WORK_DIR = '/content/drive/MyDrive/SparkleV2', '/content'
elif os.path.isdir('/kaggle/input'):
    PROJECT_DIR, WORK_DIR = '/kaggle/input/sparklev2', '/kaggle/working'
else:
    PROJECT_DIR = os.environ.get('SPARKLE_PROJECT_DIR', './SparkleV2'); WORK_DIR = '.'
assert os.path.isdir(PROJECT_DIR), f'No existe {PROJECT_DIR} (tiene que estar en "Mi unidad")'

LABELS_DIR   = f'{PROJECT_DIR}/labels'
BACKBONE_DIR = f'{PROJECT_DIR}/checkpoints_fase3_2'
EMB128_DIR   = f'{PROJECT_DIR}/embeddings_cnn'          # los de 3.2a, solo para verificar
RAW_DIR      = f'{PROJECT_DIR}/embeddings_raw1280'      # destino nuevo (Drive)
RAW_LOCAL    = f'{WORK_DIR}/embeddings_raw1280'         # staging local (rápido)
CACHE_32     = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_2_alineado.npz'
CACHE_1280   = f'{PROJECT_DIR}/dataset_final_fase3/dataset_fase3_5_raw1280.npz'
for split in ('Train', 'Validation', 'Test'):
    os.makedirs(f'{RAW_DIR}/{split}', exist_ok=True)
    os.makedirs(f'{RAW_LOCAL}/{split}', exist_ok=True)
assert os.path.isdir(LABELS_DIR), f'Falta {LABELS_DIR}'

import numpy as np
import pandas as pd
import cv2
import tensorflow as tf
import keras
from keras import layers
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm.auto import tqdm

gpus = tf.config.list_physical_devices('GPU')
print(f'TF {tf.__version__} | Keras {keras.__version__} | mediapipe {mp.__version__} | GPU: {gpus}')
if not gpus:
    print('\n*** SIN GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU ***\n')

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED)

# --- Constantes: EXACTAMENTE las de 3.2a. No tocar. ---
CROP_SIZE    = 112
N_EMB_FRAMES = 16
EMB_DIM_PROJ = 128          # solo para el assert de fidelidad
ALPHA        = 0.35
SYNC_EVERY   = 200
DTYPE_RAW    = np.float16   # §2 del protocolo

def setup_kaggle_credentials():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return 'ya configuradas'
    if IN_COLAB:
        try:
            from google.colab import userdata
            os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
            os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
            return 'Secretos de Colab'
        except Exception:
            pass
    for cand in (f'{PROJECT_DIR}/kaggle.json', '/content/drive/MyDrive/kaggle.json',
                 os.path.expanduser('~/.kaggle/kaggle.json')):
        if os.path.exists(cand):
            with open(cand) as f:
                c = json.load(f)
            os.environ['KAGGLE_USERNAME'] = c['username']; os.environ['KAGGLE_KEY'] = c['key']
            return f'kaggle.json en {cand}'
    if IN_COLAB:
        from google.colab import files
        print('No encontre credenciales. Subi tu kaggle.json:')
        up = files.upload(); name = list(up)[0]; c = json.loads(up[name])
        os.environ['KAGGLE_USERNAME'] = c['username']; os.environ['KAGGLE_KEY'] = c['key']
        return 'subida manual'
    raise RuntimeError('Sin credenciales de Kaggle')

print('Credenciales de Kaggle:', setup_kaggle_credentials())
total, used, free = shutil.disk_usage(WORK_DIR)
print(f'Disco en {WORK_DIR}: {free/1e9:.0f} GB libres de {total/1e9:.0f} GB '
      f'(DAiSEE crudo necesita ~25-30 GB)')

## 1. DAiSEE, etiquetas e índice de videos — igual que 3.2a

In [ ]:
import kagglehub
DAISEE_SLUG = 'olgaparfenova/daisee'

def find_dataset_root(base):
    for root, dirs, _ in os.walk(base):
        if {'Train', 'Validation', 'Test'}.issubset(set(dirs)):
            return root
    return base

t0 = time.time()
RAW_DATASET_DIR = find_dataset_root(kagglehub.dataset_download(DAISEE_SLUG))
print(f'DAiSEE en {RAW_DATASET_DIR}  ({(time.time()-t0)/60:.1f} min)')

def load_labels(split):
    df = pd.read_csv(f'{LABELS_DIR}/{split}Labels.csv')
    df.columns = [c.strip() for c in df.columns]
    return df

labels = {s: load_labels(s) for s in ('Train', 'Validation', 'Test')}
for s, df in labels.items():
    print(f'{s}Labels.csv: {len(df)} filas')

video_index = {}
for split in ('Train', 'Validation', 'Test'):
    for ext in ('avi', 'mp4'):
        for p in glob.glob(f'{RAW_DATASET_DIR}/{split}/**/*.{ext}', recursive=True):
            video_index[os.path.basename(p)] = p
print(f'Videos indexados: {len(video_index)}')
assert len(video_index) > 0, 'No se indexo ningun video: revisar RAW_DATASET_DIR'

## 2. FaceLandmarker y recorte facial — funciones copiadas literalmente de 3.2a

Se copian sin cambios a propósito: cualquier diferencia en el recorte invalidaría la
comparación con los embeddings existentes.

In [ ]:
import urllib.request
MODEL_URL  = ('https://storage.googleapis.com/mediapipe-models/face_landmarker/'
              'face_landmarker/float16/latest/face_landmarker.task')
MODEL_PATH = f'{WORK_DIR}/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Descargando face_landmarker.task ...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
print(f'FaceLandmarker: {os.path.getsize(MODEL_PATH)/1e6:.1f} MB')

def create_face_landmarker():
    return mp_vision.FaceLandmarker.create_from_options(mp_vision.FaceLandmarkerOptions(
        base_options=mp_python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1, min_face_detection_confidence=0.5))

def crop_face_from_landmarks(frame_bgr, landmarks, crop_size=CROP_SIZE, margin=0.25):
    h, w = frame_bgr.shape[:2]
    xs = np.array([lm.x for lm in landmarks]) * w
    ys = np.array([lm.y for lm in landmarks]) * h
    x_min, x_max, y_min, y_max = xs.min(), xs.max(), ys.min(), ys.max()
    bw, bh = x_max - x_min, y_max - y_min
    side = max(bw, bh) * (1 + margin)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    x0, y0 = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
    x1, y1 = int(min(w, cx + side / 2)), int(min(h, cy + side / 2))
    if x1 <= x0 or y1 <= y0:
        return None
    crop = cv2.resize(frame_bgr[y0:y1, x0:x1], (crop_size, crop_size), interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)

def extract_crop_from_frame(detector, frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb))
    if not result.face_landmarks:
        return None
    return crop_face_from_landmarks(frame_bgr, result.face_landmarks[0])

def sample_frames_sequential(video_path, n_frames=N_EMB_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        cap.release(); return None
    wanted = np.linspace(0, total - 1, n_frames).astype(int)
    wanted_set = {}
    for pos, idx in enumerate(wanted):
        wanted_set.setdefault(int(idx), []).append(pos)
    frames = [None] * n_frames
    i, max_idx = 0, int(wanted.max())
    while i <= max_idx:
        ret, frame = cap.read()
        if not ret:
            break
        if i in wanted_set:
            for pos in wanted_set[i]:
                frames[pos] = frame
        i += 1
    cap.release()
    return frames

## 3. Backbone — se CARGA del checkpoint, no se reconstruye

`cnn_extractor.keras` es `backbone(pooling='avg') → Dense(128)`. Se recupera el backbone y,
por separado, los pesos de la proyección aleatoria (W de 1.280×128 y su sesgo) que se usan
solo para el assert de fidelidad de la sección 5 y, más adelante, para reconstruir el brazo
R1 exactamente.

In [ ]:
EXTRACTOR_CKPT = f'{BACKBONE_DIR}/cnn_extractor.keras'
assert os.path.exists(EXTRACTOR_CKPT), (
    f'Falta {EXTRACTOR_CKPT}. Sin ese archivo NO se puede garantizar que el backbone sea el '
    'mismo que produjo los embeddings actuales, y la comparación del experimento 2 pierde '
    'validez. No sustituir por un backbone reconstruido.')

extractor_128 = keras.models.load_model(EXTRACTOR_CKPT)
extractor_128.trainable = False

# El backbone es el submodelo MobileNetV2 dentro del extractor
backbone = next(l for l in extractor_128.layers
                if isinstance(l, keras.Model) and 'mobilenet' in l.name.lower())
backbone.trainable = False
proj_layer = extractor_128.get_layer('embedding_projection')
W_rand, b_rand = [np.asarray(w) for w in proj_layer.get_weights()]

DIM_RAW = int(backbone.output_shape[-1])
print(f'Backbone recuperado: salida {DIM_RAW}-d | proyección aleatoria {W_rand.shape}')
assert DIM_RAW == 1280, f'Se esperaba 1.280-d, salió {DIM_RAW}'
assert W_rand.shape == (DIM_RAW, EMB_DIM_PROJ)

# Modelo de extracción cruda: crop -> 1280-d (sin proyectar)
inp_crop = keras.Input(shape=(CROP_SIZE, CROP_SIZE, 3), name='crop')
extractor_raw = keras.Model(inp_crop, backbone(inp_crop, training=False), name='cnn_raw1280')
extractor_raw.trainable = False

np.savez(f'{BACKBONE_DIR}/proyeccion_aleatoria_original.npz', W=W_rand, b=b_rand)
print('Proyección aleatoria original guardada (necesaria para reconstruir R1).')

## 4. Procesamiento por clip — misma lógica, salida de 1.280-d en float16

In [ ]:
def process_clip_raw(video_path, detector, extractor, n_frames=N_EMB_FRAMES):
    frames = sample_frames_sequential(video_path, n_frames)
    if frames is None:
        return None
    crops = np.zeros((n_frames, CROP_SIZE, CROP_SIZE, 3), dtype=np.float32)
    valid = np.zeros(n_frames, dtype=bool)
    for pos, frame in enumerate(frames):
        if frame is None:
            continue
        crop = extract_crop_from_frame(detector, frame)   # el frame se descarta al salir del scope
        if crop is not None:
            crops[pos] = keras.applications.mobilenet_v2.preprocess_input(crop.astype(np.float32))
            valid[pos] = True
    if not valid.any():
        return np.full((n_frames, DIM_RAW), np.nan, dtype=np.float32)
    out = np.asarray(extractor(crops, training=False)).astype(np.float32)
    out[~valid] = np.nan          # frames sin rostro -> NaN, mismo criterio que 3.2a
    return out

## 5. Assert de fidelidad — el paso que valida todo el experimento

Para 10 clips ya extraídos en 3.2a: reproyectar el 1.280-d nuevo con los pesos aleatorios
guardados tiene que reproducir el 128-d almacenado. Si falla, algo del recorte, del muestreo
o del backbone cambió, y **no hay que seguir**: el experimento 2 dejaría de ser una
comparación de proyecciones.

In [ ]:
ref_files = sorted(glob.glob(f'{EMB128_DIR}/Train/*.npz'))[:10]
assert ref_files, f'No hay embeddings de 3.2a en {EMB128_DIR}/Train para verificar'
detector = create_face_landmarker()
difs = []
try:
    for f in ref_files:
        clip_key = os.path.splitext(os.path.basename(f))[0]
        clip_id = next((c for c in (f'{clip_key}.avi', f'{clip_key}.mp4') if c in video_index), None)
        if clip_id is None:
            print(f'{clip_key}: sin video físico, se saltea'); continue
        raw = process_clip_raw(video_index[clip_id], detector, extractor_raw)
        ref = np.load(f)['embeddings']                       # (16, 128) de 3.2a
        rep = np.nan_to_num(raw) @ W_rand + b_rand           # reproyección
        m = ~np.isnan(ref).any(axis=1) & ~np.isnan(raw).any(axis=1)
        if m.sum() == 0:
            continue
        dif = float(np.abs(rep[m] - ref[m]).max()); difs.append(dif)
        print(f'{clip_key}: {m.sum():2d} frames válidos | dif. máxima {dif:.2e}')
finally:
    detector.close()

assert difs, 'No se pudo verificar ningún clip'
print(f'\nDiferencia máxima global: {max(difs):.2e}')
assert max(difs) < 1e-3, (
    f'FIDELIDAD ROTA (dif {max(difs):.2e}). El pipeline no reproduce los embeddings de 3.2a. '
    'NO continuar: revisar backbone, recorte y muestreo antes de re-extraer nada.')
print('✅ Fidelidad verificada: el 1.280-d reproyectado reproduce los 128-d de 3.2a.')
print('   ⇒ el brazo R1 del experimento 2 es exactamente el estado actual.')

## 6. Extracción completa — reanudable, con staging local y sync a Drive

Idempotente: se puede cortar y volver a correr; saltea lo ya hecho.

In [ ]:
SPLITS = ['Train', 'Validation', 'Test']

def sync_to_drive(split):
    src, dst = f'{RAW_LOCAL}/{split}', f'{RAW_DIR}/{split}'
    n = 0
    for f in os.listdir(src):
        if not os.path.exists(f'{dst}/{f}'):
            shutil.copy2(f'{src}/{f}', f'{dst}/{f}'); n += 1
    return n

def already_done_ids(split):
    done = set()
    for base in (f'{RAW_DIR}/{split}', f'{RAW_LOCAL}/{split}'):
        if os.path.isdir(base):
            done |= {os.path.splitext(f)[0] for f in os.listdir(base) if f.endswith('.npz')}
    return done

for SPLIT in SPLITS:
    out_dir  = f'{RAW_LOCAL}/{SPLIT}'
    log_path = f'{RAW_DIR}/{SPLIT}_processing_log.csv'
    log_rows = pd.read_csv(log_path).to_dict('records') if os.path.exists(log_path) else []
    done = already_done_ids(SPLIT)
    pendientes = [r for _, r in labels[SPLIT].iterrows()
                  if os.path.splitext(r['ClipID'])[0] not in done]
    print(f'\n=== {SPLIT}: {len(done)} ya procesados, {len(pendientes)} pendientes '
          f'de {len(labels[SPLIT])} ===')
    if not pendientes:
        sync_to_drive(SPLIT); continue

    detector = create_face_landmarker()
    procesados = 0
    try:
        for row in tqdm(pendientes, desc=SPLIT):
            clip_id = row['ClipID']; clip_key = os.path.splitext(clip_id)[0]
            if clip_id not in video_index:
                log_rows.append({'ClipID': clip_id, 'status': 'sin_video_fisico'}); continue
            try:
                emb = process_clip_raw(video_index[clip_id], detector, extractor_raw)
                if emb is None:
                    log_rows.append({'ClipID': clip_id, 'status': 'video_ilegible'}); continue
                pct_nan = float(np.isnan(emb).any(axis=1).mean() * 100)
                np.savez_compressed(f'{out_dir}/{clip_key}.npz',
                                    embeddings=emb.astype(DTYPE_RAW))
                log_rows.append({'ClipID': clip_id, 'status': 'ok', 'pct_nan': pct_nan})
                procesados += 1
            except Exception as e:
                log_rows.append({'ClipID': clip_id, 'status': f'error: {type(e).__name__}: {e}'})
            if procesados and procesados % SYNC_EVERY == 0:
                n = sync_to_drive(SPLIT)
                pd.DataFrame(log_rows).to_csv(log_path, index=False)
                tqdm.write(f'  [checkpoint] {procesados} clips | {n} archivos nuevos a Drive')
    finally:
        detector.close()
        n = sync_to_drive(SPLIT)
        pd.DataFrame(log_rows).to_csv(log_path, index=False)
        print(f'{SPLIT}: sincronizados {n} archivos a Drive. Log en {log_path}')

for SPLIT in SPLITS:
    n_drive = len(glob.glob(f'{RAW_DIR}/{SPLIT}/*.npz'))
    log_path = f'{RAW_DIR}/{SPLIT}_processing_log.csv'
    if os.path.exists(log_path):
        st = pd.read_csv(log_path)['status'].astype(str).str.split(':').str[0]
        print(f'{SPLIT}: {n_drive} .npz en Drive | estados: {dict(st.value_counts())}')
    else:
        print(f'{SPLIT}: {n_drive} .npz en Drive')

## 7. Caché consolidada, alineada al orden de la caché de 3.2b

La alineación se hereda de `dataset_fase3_2_alineado.npz`: se recorre **su** orden de ClipIDs
y se apila el 1.280-d correspondiente. Así la geometría, las etiquetas y los IDs del
experimento 2 son **bit a bit los mismos** que en 3.3, y lo único distinto es la apariencia.

In [ ]:
assert os.path.exists(CACHE_32), f'Falta {CACHE_32} (la caché alineada de 3.2b)'
d32 = np.load(CACHE_32, allow_pickle=True)

def apilar(split, ids):
    ids = [str(c) for c in ids]
    faltan = [c for c in ids if not os.path.exists(f'{RAW_DIR}/{split}/{c}.npz')]
    assert not faltan, (f'{len(faltan)} clips de la caché de 3.2b sin 1.280-d en {split} '
                        f'(ej. {faltan[:3]}). Reanudar la sección 6 antes de consolidar.')
    out = np.empty((len(ids), N_EMB_FRAMES, DIM_RAW), dtype=DTYPE_RAW)
    for i, c in enumerate(tqdm(ids, desc=f'consolidando {split}')):
        out[i] = np.load(f'{RAW_DIR}/{split}/{c}.npz')['embeddings']
    return out

Xr_tr = apilar('Train', d32['id_tr'])
Xr_va = apilar('Validation', d32['id_va'])
Xr_te = apilar('Test', d32['id_te'])
print(f'\nTrain {Xr_tr.shape} | Validation {Xr_va.shape} | Test {Xr_te.shape} | dtype {Xr_tr.dtype}')

os.makedirs(os.path.dirname(CACHE_1280), exist_ok=True)
np.savez(CACHE_1280, Xr_tr=Xr_tr, Xr_va=Xr_va, Xr_te=Xr_te,
         id_tr=d32['id_tr'], id_va=d32['id_va'], id_te=d32['id_te'], dim_raw=DIM_RAW)
print(f'Caché guardada: {CACHE_1280} ({os.path.getsize(CACHE_1280)/1e6:.0f} MB)')

## 8. Control de calidad — comparable con §7.3 del registro

Los umbrales de referencia de 3.2a sobre 300 clips de Train fueron: 0 % de frames sin rostro,
norma media 13,93, desvío entre clips 0,65, desvío dentro del clip 0,32 y similitud coseno
media entre clips distintos 0,726 (alarma si > 0,98). Acá el espacio es de 1.280-d, así que
los valores absolutos no son comparables uno a uno, pero **la relación entre variabilidad
entre clips y dentro del clip sí lo es**, y la similitud coseno no debería ser mayor.

In [ ]:
M = Xr_tr[:300].astype(np.float32)
clip_mean = np.nanmean(M, axis=1)
norms = np.linalg.norm(np.nan_to_num(clip_mean), axis=1)
inter = float(np.nanstd(clip_mean, axis=0).mean())
intra = float(np.nanstd(M, axis=1).mean())
V = np.nan_to_num(clip_mean)
V = V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-9)
S = V @ V.T
cos_med = float(S[np.triu_indices(len(V), k=1)].mean())

print(f'Frames sin rostro      : {np.isnan(M).any(axis=2).mean()*100:.2f} %')
print(f'Norma media por clip   : {norms.mean():.3f}')
print(f'Desvío ENTRE clips     : {inter:.4f}')
print(f'Desvío DENTRO del clip : {intra:.4f}')
print(f'Similitud coseno media : {cos_med:.3f}   (alarma si > 0,98)')
assert cos_med < 0.98, 'Embeddings degenerados: todos los clips dan casi el mismo vector.'
print('\nRelación entre/dentro:', round(inter / max(intra, 1e-9), 2),
      '(en 3.2a fue 0,6547/0,3194 = 2,05)')

## 9. Checkpoint de 3.5a

Antes de pasar a 3.5b, confirmar:

1. El assert de fidelidad de §5 pasó. **Si no pasó, no seguir** — el experimento 2 dejaría de
   ser una comparación de proyecciones y habría que diagnosticar el pipeline primero.
2. Los tres splits cerraron con 0 errores en los logs (comparar con los 8.571 clips y 0 fallos
   de 3.2a, §7.2 del registro).
3. La caché `dataset_fase3_5_raw1280.npz` existe y sus shapes coinciden en número de clips con
   la caché de 3.2b.
4. La similitud coseno media quedó lejos de 0,98.

Registrar en `Fase3_Registro_de_Resultados.md`: tiempo de extracción, estados de los logs,
métricas de QC y el valor del assert de fidelidad. Después correr
`Sparkle_V2_Fase3_5b_Brazos_Proyeccion.ipynb`.